In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3

conn = sqlite3.connect('/content/drive/MyDrive/churn-prediction-project/churn.db')
df = pd.read_sql("SELECT * FROM customers", conn)

# Fix Total Charges (same issue as before)
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

print(df.shape)
df.columns.tolist()

Mounted at /content/drive
(7043, 33)


['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [2]:
print(df.columns.tolist())

['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


In [3]:
# Target variable: 1 = churned, 0 = stayed
target = df['Churn Value']

# Drop leakage, ID, and location columns, plus the duplicate target column
cols_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
                 'Lat Long', 'Latitude', 'Longitude',
                 'Churn Label', 'Churn Score', 'Churn Reason', 'Churn Value']

# Only drop columns that actually exist (in case names differ slightly)
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
features_df = df.drop(columns=cols_to_drop)

print("Remaining columns:", features_df.columns.tolist())
print("Shape:", features_df.shape)

Remaining columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'CLTV']
Shape: (7043, 20)


In [4]:
# Check which columns are text vs numbers
categorical_cols = features_df.select_dtypes(include='object').columns.tolist()
numeric_cols = features_df.select_dtypes(include=['int64','float64']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

# One-hot encode the categorical columns
X = pd.get_dummies(features_df, columns=categorical_cols, drop_first=True)
y = target  # 0 = stayed, 1 = churned

print("X shape after encoding:", X.shape)

Categorical columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']
Numeric columns: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']
X shape after encoding: (7043, 31)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (5634, 31) Test: (1409, 31)


In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [7]:
import os, joblib

save_path = '/content/drive/MyDrive/churn-prediction-project/processed'
os.makedirs(save_path, exist_ok=True)

X_train.to_csv(f'{save_path}/X_train.csv', index=False)
X_test.to_csv(f'{save_path}/X_test.csv', index=False)
y_train.to_csv(f'{save_path}/y_train.csv', index=False)
y_test.to_csv(f'{save_path}/y_test.csv', index=False)
joblib.dump(scaler, f'{save_path}/scaler.pkl')

print("Saved! X_train columns:", X_train.shape[1])

Saved! X_train columns: 31
